# D-FINE-N reproduction


In [1]:
import subprocess

gpu = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,driver_version,memory.total",
        "--format=csv,noheader",
    ],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(gpu)
if not gpu.startswith("Tesla T4,"):
    raise RuntimeError("select a Tesla T4 runtime for this reproduction")

Tesla T4, 580.82.07, 15360 MiB


In [2]:
import os
from pathlib import Path

installer = Path("/tmp/uv-install.sh")
subprocess.run(
    [
        "curl",
        "-LsSf",
        "https://astral.sh/uv/0.12.23/install.sh",
        "-o",
        str(installer),
    ],
    check=True,
)
env = os.environ.copy()
env["UV_UNMANAGED_INSTALL"] = "/usr/local/bin"
subprocess.run(["sh", str(installer)], check=True, env=env)

uv_version = subprocess.run(
    ["uv", "--version"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(uv_version)
if not uv_version.startswith("uv 0.12.23 "):
    raise RuntimeError(f"unexpected uv version: {uv_version}")

uv 0.12.23 (x86_64-unknown-linux-gnu)


In [3]:
import shutil

repository = "https://github.com/aekn/smallpotato.git"
revision = "071a4a66acd9e8b3ae3b523110a6becfe208b304"
repo = Path("/content/smallpotato")

if repo.exists():
    shutil.rmtree(repo)

subprocess.run(["git", "init", "-q", str(repo)], check=True)
subprocess.run(
    ["git", "-C", str(repo), "remote", "add", "origin", repository],
    check=True,
)
subprocess.run(
    [
        "git",
        "-C",
        str(repo),
        "fetch",
        "-q",
        "--depth=1",
        "--no-tags",
        "origin",
        revision,
    ],
    check=True,
)
subprocess.run(
    ["git", "-C", str(repo), "checkout", "-q", "--detach", "FETCH_HEAD"],
    check=True,
)

actual = subprocess.run(
    ["git", "-C", str(repo), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if actual != revision:
    raise RuntimeError(f"expected {revision}, got {actual}")

status = subprocess.run(
    ["git", "-C", str(repo), "status", "--porcelain"],
    check=True,
    capture_output=True,
    text=True,
).stdout
if status:
    raise RuntimeError("repository checkout is dirty")

os.chdir(repo)
print(f"{repo} @ {actual}")

/content/smallpotato @ 071a4a66acd9e8b3ae3b523110a6becfe208b304


In [4]:
subprocess.run(["uv", "sync", "--locked"], cwd=repo, check=True)
print("environment ready")

environment ready


In [5]:
process = subprocess.Popen(
    ["uv", "run", "python", "-m", "reproductions.dfine.reproduce"],
    cwd=repo,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)

returncode = process.wait()
if returncode:
    raise subprocess.CalledProcessError(returncode, process.args)

smallpotato - reproduce dfine

model       dfine-n
dataset     coco/val2017
run         /content/smallpotato/runs/dfine/20261009T214326Z

prepare     inputs
check       source                      ok
check       data                        ok
check       checkpoint                  ok
sync        environment
device      Tesla T4
check       environment                 ok
test         1/79    1.3%   eta 15m33s
test        11/79   13.9%   eta 2m55s
test        21/79   26.6%   eta 2m06s
test        31/79   39.2%   eta 1m39s
test        41/79   51.9%   eta 1m16s
test        51/79   64.6%   eta 55s
test        61/79   77.2%   eta 35s
test        71/79   89.9%   eta 16s
test        79/79  100.0%
warning     upstream emitted 1 warning line(s); see run.log
coco        load
coco        evaluate
coco        accumulate
coco        summarize

metric         AP      AP50      AP75       APs       APm       APl
             0.4280    0.6028    0.4545    0.2288    0.4679    0.6215
metric        AR1  

In [6]:
runs = repo / "runs" / "dfine"
run_dir = max(
    (path for path in runs.iterdir() if path.is_dir()),
    key=lambda path: path.name,
)
print(run_dir)

for name in ("summary.json", "provenance.json"):
    print(f"\n{name}")
    print((run_dir / name).read_text(encoding="utf-8"), end="")

/content/smallpotato/runs/dfine/20261009T214326Z

summary.json
{
  "durations_seconds": {
    "environment": 69.88455113700002,
    "evaluation": 122.20201793799993,
    "inference": 152.73955424499997,
    "inputs": 77.922262945,
    "total": 422.79618221
  },
  "finished_at": "2026-10-09T21:50:29.219188Z",
  "metrics": {
    "ap": 0.4279972400671271,
    "ap50": 0.6028099059884774,
    "ap75": 0.4545386692428566,
    "ap_large": 0.6215192886300667,
    "ap_medium": 0.4678795208625124,
    "ap_small": 0.22882582627319026,
    "ar1": 0.34256664197573444,
    "ar10": 0.5711430940727233,
    "ar100": 0.634649524097191,
    "ar_large": 0.8567128147266587,
    "ar_medium": 0.6975149377867146,
    "ar_small": 0.3979374302915091
  },
  "prediction": {
    "path": "pred.bbox.json",
    "sha256": "ab53c27e191d788533d2250a205f69bdce9b3d216add073431f820752569ab25",
    "size": 219283872
  },
  "reference": {
    "max_abs_error": 0.0,
    "metrics": {
      "ap": 0.4279972400671271,
      "ap50":

In [7]:
from google.colab import files

archive_base = Path("/content") / f"dfine-{run_dir.name}"
run_archive = shutil.make_archive(
    str(archive_base),
    "gztar",
    root_dir=run_dir.parent,
    base_dir=run_dir.name,
)
print(run_archive)
files.download(run_archive)

/content/dfine-20261009T214326Z.tar.gz


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>